In [6]:
from pyspark.sql import SparkSession
from delta import *

# Configurando o Spark com suporte nativo ao Delta Lake
builder = SparkSession.builder.appName("DeltaLakeFootball") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.2.0")


# Iniciando a sessão configurada
spark = configure_spark_with_delta_pip(builder).getOrCreate()

print("Sucesso! A sessão Spark com Delta Lake está ativa.")

Sucesso! A sessão Spark com Delta Lake está ativa.


In [7]:
# 1. Definição do caminho onde os dados serão salvos (camada Silver)
path_delta = "../data/delta/fato_evento"

# 2. Criando dados iniciais de uma partida (Simulação: Flamengo x Vasco)
# id_evento, id_partida, id_jogador, tipo_evento, minuto_jogo, revisado_var
data = [
    (1, 101, 10, "Gol", 15, False),             # Gol do camisa 10
    (2, 101, 5, "Cartão Amarelo", 30, False),    # Falta do volante
    (3, 101, 9, "Gol", 44, False)              # Gol do centroavante
]

columns = ["id_evento", "id_partida", "id_jogador", "tipo_evento", "minuto_jogo", "revisado_var"]

# 3. Criando o DataFrame do Spark
df_inicial = spark.createDataFrame(data, columns)

# 4. Salvando no formato Delta (Operação: INSERT inicial)
df_inicial.write.format("delta").mode("overwrite").save(path_delta)

# 5. Lendo e exibindo a tabela para conferir
df_check = spark.read.format("delta").load(path_delta)
df_check.show()

+---------+----------+----------+--------------+-----------+------------+
|id_evento|id_partida|id_jogador|   tipo_evento|minuto_jogo|revisado_var|
+---------+----------+----------+--------------+-----------+------------+
|        2|       101|         5|Cartão Amarelo|         30|       false|
|        3|       101|         9|           Gol|         44|       false|
|        1|       101|        10|           Gol|         15|       false|
+---------+----------+----------+--------------+-----------+------------+



In [8]:
from delta.tables import *

# 1. Criar o objeto DeltaTable (essencial para comandos de atualização e exclusão)
delta_table = DeltaTable.forPath(spark, path_delta)

# 2. Cenário VAR 1: Corrigindo o autor do gol (id_evento = 1)
print(">>> VAR analisando o primeiro gol...")
delta_table.update(
    condition = "id_evento = 1",
    set = { "id_jogador": "99", "revisado_var": "True" }
)

# 3. Cenário VAR 2: Anulando o gol por impedimento (id_evento = 3)
print(">>> VAR anulando o segundo gol por impedimento...")
delta_table.delete("id_evento = 3")

# 4. Exibir a tabela final atualizada
print("\nPlacar Final atualizado pelo VAR:")
delta_table.toDF().orderBy("id_evento").show()

>>> VAR analisando o primeiro gol...
>>> VAR anulando o segundo gol por impedimento...

Placar Final atualizado pelo VAR:
+---------+----------+----------+--------------+-----------+------------+
|id_evento|id_partida|id_jogador|   tipo_evento|minuto_jogo|revisado_var|
+---------+----------+----------+--------------+-----------+------------+
|        1|       101|        99|           Gol|         15|        true|
|        2|       101|         5|Cartão Amarelo|         30|       false|
+---------+----------+----------+--------------+-----------+------------+



In [9]:
# Viagem no Tempo: Consultando os dados originais (antes do VAR)
print(">>> Consultando a Versão 0 (Antes da intervenção do VAR):")
df_v0 = spark.read.format("delta").option("versionAsOf", 0).load(path_delta)
df_v0.orderBy("id_evento").show()

>>> Consultando a Versão 0 (Antes da intervenção do VAR):


+---------+----------+----------+--------------+-----------+------------+
|id_evento|id_partida|id_jogador|   tipo_evento|minuto_jogo|revisado_var|
+---------+----------+----------+--------------+-----------+------------+
|        1|       101|        10|           Gol|         15|       false|
|        2|       101|         5|Cartão Amarelo|         30|       false|
|        3|       101|         9|           Gol|         44|       false|
+---------+----------+----------+--------------+-----------+------------+

